In [ ]:
import pandas as pd
import numpy as np

CLEAN_PATH = "data/cleaned/cpi_clean.csv"
MONTHS = ["January","February","March","April","May","June","July","August","September","October","November","December"]

cpi = pd.read_csv(CLEAN_PATH, parse_dates=["period"])
cpi["month"] = pd.Categorical(cpi["month"], categories=MONTHS, ordered=True)   # CSV forgets category order
print(cpi.shape)
cpi.head(3)

In [ ]:
from pathlib import Path

Path("data/reference").mkdir(parents=True, exist_ok=True)

zones.to_csv(
    "data/reference/state_regions.csv",
    index=False
)

In [ ]:
from pathlib import Path

output = Path(
    r"C:\Users\Admin\OneDrive\Documents\GitHub\cpi-infaltion-analysis\data\Reference"
)

output.mkdir(parents=True, exist_ok=True)

zones.to_csv(output / "state_regions.csv", index=False)

print("Saved:", output / "state_regions.csv")

In [ ]:
print(zones.head())
print(zones.columns.tolist())
print(zones.shape)

## Part 1 – Missing data: investigate, decide, document
Three gaps exist in this file. Count each one in code before writing anything down.

In [ ]:
# Gap 1: inflation is blank in 2025
cpi.groupby("year").agg(rows=("inflation", "size"), missing_inflation=("inflation", lambda s: s.isna().sum()))

In [ ]:
# Gap 2: some state/sector combinations have fewer rows than others
counts = cpi.groupby(["state", "sector"], observed=True).size().unstack()
counts[counts.nunique(axis=1) > 1]

In [ ]:
# Gap 3: the 'code' column is blank for which divisions?
cpi[cpi["code"].isna()]["division"].unique()

## Part 2 – Merging: add a region lookup
A **merge** adds columns from one table to another by matching a key (here, `state`), like VLOOKUP but for whole tables.

**Source for the grouping:** the five Zonal Councils (Northern, Central, Eastern, Western, Southern) plus the North Eastern Council
for the eight north-eastern states. Andaman and Nicobar Islands and Lakshadweep belong to no zonal council
(they attend the Southern Zonal Council as special invitees), so they are grouped as "Islands" here.
This is a **project grouping, not an official statistical classification**. Verify it against the Ministry of Home Affairs
page and cite it in your README.

In [ ]:
zones = pd.DataFrame({
    "state": [
        "Chandigarh", "Delhi", "Haryana", "Himachal Pradesh", "Jammu and Kashmir", "Ladakh", "Punjab", "Rajasthan",
        "Chhattisgarh", "Madhya Pradesh", "Uttar Pradesh", "Uttarakhand",
        "Bihar", "Jharkhand", "Odisha", "West Bengal",
        "Goa", "Gujarat", "Maharashtra", "Dadra and Nagar Haveli and Daman and Diu",
        "Andhra Pradesh", "Karnataka", "Kerala", "Puducherry", "Tamil Nadu", "Telangana",
        "Arunachal Pradesh", "Assam", "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Sikkim", "Tripura",
        "Andaman and Nicobar Islands", "Lakshadweep",
    ],
    "region_group": (["Northern"] * 8 + ["Central"] * 4 + ["Eastern"] * 4 + ["Western"] * 4
                     + ["Southern"] * 6 + ["North-Eastern"] * 8 + ["Islands"] * 2),
})
print(len(zones), "rows;", zones["state"].is_unique and "state is unique" or "DUPLICATE states")

In [ ]:
def merge_zones(df, zones):
    """Left-merge the region lookup onto the data and report anything that did not match."""
    m = df.merge(zones, on="state", how="left", validate="many_to_one", indicator=True)
    unmatched = sorted(m.loc[m["_merge"] == "left_only", "state"].unique())
    print("Unmatched state names:", unmatched)
    return m.drop(columns="_merge")

merged = merge_zones(cpi, zones)

In [ ]:
# TODO: fix zones here


merged = merge_zones(cpi, zones)
still_unmatched = set(merged.loc[merged["region_group"].isna(), "state"]) - {"All India"}
print("PASS" if not still_unmatched else f"FAIL - still unmatched: {sorted(still_unmatched)}")

## Part 3 – Grouping
`groupby` splits the table into groups, applies a calculation to each, and combines the results.

**Worked example.** September 2026, CPI (General), Combined sector, states only. Look at the `count` column:
a group of 2 states and a group of 8 states are not equally reliable, and the mean treats every state equally,
whatever its population. **This is an unweighted mean of states, not a regional inflation rate.**

In [ ]:
sept = merged[(merged["year"] == 2026) & (merged["month"] == "September") & (merged["sector"] == "Combined")
              & (merged["division"] == "CPI (General)") & (~merged["is_aggregate"])]

sept.groupby("region_group")["inflation"].agg(["mean", "median", "min", "max", "count"]).round(2)

In [ ]:
# TODO 1
state_summary = None

# TODO 2

# TODO 3

# TODO 4

## Part 4 – Save a Tableau-ready file
Tableau works best with **long format**: one row per observation, one column per attribute. This file already is.

In [ ]:
import os
final = merged.copy()
final.loc[final["state"] == "All India", "region_group"] = "National"   # decision logged in cleaning_log.md
os.makedirs("data/cleaned", exist_ok=True)
final.to_csv("data/cleaned/cpi_tableau_long.csv", index=False)
print(final.shape, final["region_group"].value_counts(dropna=False).to_dict())